# Buzzlytics — Build the Datasets (run once)

Buzzlytics uses a **two-stage** model, so this notebook builds **two** datasets and saves both to
your Google Drive. You run it once; afterwards you only ever touch the training notebook.

| Output zip | Feeds | Contents |
|-----------|-------|----------|
| **`bee_dataset.zip`** | Stage-1 detector | VnPollenBee boxes — classes `bee`, `pollen_bee` |
| **`varroa_cls.zip`** | Stage-2 classifier | VarroaDataset crops — folders `healthy/`, `varroa/` |

### Data sources (cite these)
- **VnPollenBee** — Vietnamese hive-entrance detection set (HUST ComVis), LabelMe polygon
  annotations with embedded images. Gives real per-bee bounding boxes.
- **VarroaDataset** — TU Wien, Zenodo `10.5281/zenodo.4085044`, CC-BY-4.0. 13.5k single-bee
  *classification* crops labelled healthy/infected — **no** boxes, which is exactly why varroa is a
  separate classifier and not a detection class.

> **Run all.** Approve the two Google pop-ups (Drive API auth, then Drive mount).

## 0 · Configuration
`VARROA_LIMIT` caps how many varroa crops we keep, sampled class-balanced. The raw set is heavily
skewed (~2.4 healthy : 1 infected) and 13.5k crops would swamp the ~2k detection images, so we trim
to a balanced subset — **mention this sampling choice in the report**.

In [ ]:
REPO_URL = "https://github.com/okayxsh/Buzzlytics---CSCI435.git"
REPO_DIR = "/content/Buzzlytics---CSCI435"
VPB_FOLDER_ID = "1fdEcu7CNmEkVAamu9wh_Ppw_-uW3VNY1"   # VnPollenBee Drive folder
VARROA_LIMIT = 3000   # cap varroa crops (class-balanced). None = all 13.5k

## 1 · Install deps + clone the repo
The dataset converters live in `datasets/prepare_dataset.py`, so we clone the project (and `git pull`
on re-runs to pick up any fixes). `gdown`/`pyyaml` support the download + config writing.

In [ ]:
!pip -q install gdown pyyaml
import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

## 2 · Download VnPollenBee via the Drive API
VnPollenBee lives in a shared Drive folder whose sub-folders hold **hundreds** of LabelMe JSON files.
`gdown`'s folder scraper hard-caps at 50 files per folder, so we use the **official Drive API**
(authenticated, paginated) and download in parallel threads. Each JSON embeds its image as base64,
so the JSONs are all we need.

In [ ]:
import io, shutil, threading
from concurrent.futures import ThreadPoolExecutor
from google.colab import auth
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
auth.authenticate_user()
VPB_DIR = "/content/vnpollenbee"

def _api():
    # One client per thread — googleapiclient's http transport is not thread-safe.
    return gbuild("drive", "v3", cache_discovery=False)

def _children(api, fid):
    items, tok = [], None
    while True:
        r = api.files().list(q=f"'{fid}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)", pageSize=1000,
            pageToken=tok, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        items += r["files"]; tok = r.get("nextPageToken")
        if not tok: break
    return items

def _walk(api, fid, dest):           # recurse the folder tree -> list of (file_id, local_path)
    os.makedirs(dest, exist_ok=True)
    jobs = []
    for f in _children(api, fid):
        if f["mimeType"] == "application/vnd.google-apps.folder":
            jobs += _walk(api, f["id"], os.path.join(dest, f["name"]))
        elif f["name"].lower().endswith(".json"):
            jobs.append((f["id"], os.path.join(dest, f["name"])))
    return jobs

_tl = threading.local()
def _get(job):                        # download one file (thread-local client)
    if not hasattr(_tl, "api"): _tl.api = _api()
    fid, path = job
    dl = MediaIoBaseDownload(io.FileIO(path, "wb"), _tl.api.files().get_media(fileId=fid))
    done = False
    while not done: _, done = dl.next_chunk()

shutil.rmtree(VPB_DIR, ignore_errors=True)
print("Enumerating VnPollenBee folder...")
jobs = _walk(_api(), VPB_FOLDER_ID, VPB_DIR)
print(f"  {len(jobs)} JSONs -> downloading in parallel...")
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_get, jobs))
print(f"  pulled {len(jobs)} JSON files")

## 3 · Convert both datasets
Runs the converters **in-process** (so any error surfaces here, not hidden in a subprocess):
- `prepare_vnpollenbee` → LabelMe polygons to YOLO boxes (`nonpollenbee`→`bee`, `pollenbee`→`pollen_bee`).
- `prepare_varroa` → downloads the Zenodo crops and sorts them into a `healthy/`,`varroa/` ImageFolder.
- `write_data_yaml` → the 2-class detection config.

In [ ]:
import sys
from pathlib import Path
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
from datasets.prepare_dataset import (
    prepare_vnpollenbee, prepare_varroa, write_data_yaml, _ensure_dirs)

RAW = Path("datasets/raw"); DET = Path("datasets/data"); CLS = Path("datasets/varroa_cls")
shutil.rmtree(DET, ignore_errors=True); shutil.rmtree(CLS, ignore_errors=True)
RAW.mkdir(parents=True, exist_ok=True); _ensure_dirs(DET)

n_det = prepare_vnpollenbee(None, RAW, DET, local_dir=VPB_DIR)   # stage-1 detection
n_cls = prepare_varroa(RAW, CLS, limit=VARROA_LIMIT)             # stage-2 classification
write_data_yaml(DET)
print(f"\nDetection: {n_det} images (bee/pollen). Varroa-cls: {n_cls} crops.")

## 4 · Sanity check (quote these numbers in the report)
Detection instances per class and image counts per split, plus the varroa crop counts. Confirms the
datasets built correctly and documents the class balance you trained on.

In [ ]:
from collections import Counter
c = Counter()
for txt in Path("datasets/data").rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip(): c[int(line.split()[0])] += 1
print("detection instances per class (0=bee, 1=pollen):", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    d = Path("datasets/data") / s / "images"
    print(f"  {s}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")
print("varroa classification crops:")
for split in ["train", "val", "test"]:
    for lab in ["healthy", "varroa"]:
        p = Path("datasets/varroa_cls") / split / lab
        print(f"  {split}/{lab}: {len(list(p.glob('*'))) if p.is_dir() else 0}")

## 5 · Zip both + save to Google Drive
Writes `bee_dataset.zip` and `varroa_cls.zip` to your Drive. Then **share-link each** (right-click →
Share → *Anyone with the link* → Copy link) and paste both into the training notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DET_ZIP = '/content/drive/MyDrive/bee_dataset.zip'   # detection (stage 1)
CLS_ZIP = '/content/drive/MyDrive/varroa_cls.zip'    # classification (stage 2)
shutil.make_archive(DET_ZIP[:-4], 'zip', root_dir='datasets', base_dir='data')
shutil.make_archive(CLS_ZIP[:-4], 'zip', root_dir='datasets', base_dir='varroa_cls')
print(f"Saved {DET_ZIP}  ({os.path.getsize(DET_ZIP)/1e6:.0f} MB)")
print(f"Saved {CLS_ZIP}  ({os.path.getsize(CLS_ZIP)/1e6:.0f} MB)")
print("\nShare-link both and paste them into colab_train.ipynb (DETECT_URL, VARROA_URL).")